# Data Preparation — Training Manifest with Named Entities

In [ ]:
!pip install -U spacy transformers torch torchvision jiwer pandas tqdm torchcodec torchaudio
!python -m spacy download en_core_web_trf   # or de_dep_news_trf fr_dep_news_trf es_dep_news_trf

## Load the Kaldi Dataset

In [ ]:
import torchaudio
from IPython.display import Audio, display
from tqdm import tqdm

# all datasets provide train_nodev (train), train_dev (dev) and test splits
train_dataset_paths = [
    ("/training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/train_nodev", "voxpopuli"),
    ("/training-1/asr/kaldi_data/cv19/en/train", "commonvoice"),
    ("/training-1/asr/kaldi_data/singleword/en/train", "commonvoice_singlewords"),
    ("/training-1/asr/kaldi_data/mls/en/train", "librispeech"),
                ]# other languages *_de *_fr *_es

dev_dataset_paths = [
    ("/training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/train_dev", "voxpopuli"),
    ("/training-1/asr/kaldi_data/cv19/en/dev", "commonvoice"),
    ("/training-1/asr/kaldi_data/singleword/en/dev", "commonvoice_singlewords"),
    ("/training-1/asr/kaldi_data/mls/en/dev", "librispeech"),
]

def read_split(dataset_path):
    samples = []
    wav_scp_dict = {}
    with open(f"{dataset_path}/wav.scp", encoding="UTF-8") as rf:
        for line in rf:
            id, wav_path = line.strip().split("\t")
            wav_scp_dict[id] = wav_path

        
    text_dict = {}
    with open(f"{dataset_path}/text", encoding="UTF-8") as rf:
        for line in rf:
            try:
                id, text = line.strip().split("\t")
                text_dict[id] = text
            except Exception as ex:
                continue
    for id, wav_path in wav_scp_dict.items():
        if id in text_dict:
            text = text_dict[id]
            samples.append({"audio" : wav_path, "text" : text})
    return samples

dataset = []       # train split
dev_dataset = []   # dev split

for dataset_root, dataset_title in train_dataset_paths:
    dataset.extend(read_split(dataset_root))

for dataset_root, dataset_title in train_dataset_paths:
    dev_dataset.extend(read_split(dataset_root))

print(len(dataset), "train samples,", len(dev_dataset), "dev samples")

#helper function: load audio and make sure it is at 16kHz
def load_audio(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)

    waveform = torchaudio.functional.resample(
        waveform, sample_rate, 16000
    )

    waveform = waveform.mean(dim=0)
    return waveform

first_entry = dataset[0]
print(first_entry)
print("-"*10)
print(first_entry["text"])
audio = load_audio(first_entry["audio"])
display(Audio(audio, rate=16000))

## Named Entity Extraction

In [ ]:
import spacy

nlp = spacy.load("en_core_web_trf")   # or de_dep_news_trf fr_dep_news_trf es_dep_news_trf

def extract_entities(text):
    return [e.text for e in nlp(text).ents]

def build_prompt(text):
    seen = []
    for e in extract_entities(text):
        if e not in seen:
            seen.append(e)
    if not seen:
        return None
    return "Vocabulary: " + ", ".join(seen) + "."

print(extract_entities(first_entry["text"]))
print(build_prompt(first_entry["text"]))


## Write the Training and Dev Manifests

In [ ]:
import json
from pathlib import Path
from tqdm import tqdm
import random
MANIFEST_DIR = Path("asr_finetune")
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_TRAIN = MANIFEST_DIR / "manifest_train.jsonl"
MANIFEST_DEV = MANIFEST_DIR / "manifest_dev.jsonl"

#TODO: add manifest writing



## Inspect the Manifests

In [ ]:
for path in [MANIFEST_TRAIN, MANIFEST_DEV]:
    print("=" * 70)
    print(path)
    with open(path, encoding="UTF-8") as rf:
        for i, line in enumerate(rf):
            if i >= 5:
                break
            print(json.loads(line))